In [1]:
# %%

#------------------------------------------------ Begin_Librairie ----------------------------------------

from bs4 import BeautifulSoup
import datetime
import pandas as pd
from pandas import ExcelWriter
from selenium import webdriver
from selenium.webdriver.common.by import By
from webdriver_manager.chrome import ChromeDriverManager
# import bvdpdf
from time import sleep
import os
from selenium.common.exceptions import NoSuchElementException
from selenium.common.exceptions import ElementClickInterceptedException
from selenium.webdriver.support import expected_conditions as EC
import re
import requests
from bs4 import BeautifulSoup
import urllib3
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
# %%

#------------------------------------------------ Begin_ fileName ----------------------------------------

print("Running KY CIMA Web Scraping Tool v.1.0")

now=datetime.datetime.now()
filename= 'KY CIMA SQL Ready {}.xlsx'.format(str(now).replace(":",".")[:-7])
regulatorName = 'KY CIM'
scriptfolder = f"C:\\Users\\wuj1\\OneDrive - Moody's\\Desktop\\Regulator\\{regulatorName}"
#scriptfolder = os.path.dirname(os.path.abspath(__file__))
os.chdir(scriptfolder)
tempfolder = os.path.join(scriptfolder, 'tempfolder')

if os.path.exists(tempfolder):
    for rem_file in os.listdir(tempfolder):
        os.remove(os.path.join(tempfolder, rem_file))
else:
    os.mkdir(tempfolder)
# %%

#------------------------------------------------ Begin_chromedriver ----------------------------------------

chromeOptions = webdriver.ChromeOptions()
prefs = {"plugins.always_open_pdf_externally": True,
		 "download.prompt_for_download": False,
		 "download.default_directory" : tempfolder}
chromeOptions.add_experimental_option("prefs",prefs)
chromeOptions.add_experimental_option("excludeSwitches", ["enable-automation"])
user_agent = "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"
chromeOptions.add_argument(f"user-agent={user_agent}")

driver = webdriver.Chrome(options=chromeOptions)
driver.maximize_window()
#------------------------------------------------ Begin_Fouction ----------------------------------------

def bourange_same_length_array(sqldict) :

    maxlen = len(sqldict['ListProcessDate'])
    for key, val in sqldict.items():
        if len(sqldict[key]) != maxlen:
            empty = []
            total_empty = maxlen - len(sqldict[key])

            for i in range(total_empty):
                empty.append('')
            sqldict[key]=sqldict[key]+empty
    return sqldict

Running KY CIMA Web Scraping Tool v.1.0


In [2]:

def make_session():
    s = requests.Session()
    retry = Retry(
        total=5,
        backoff_factor=1.0,
        status_forcelist=(429, 500, 502, 503, 504),
        allowed_methods=("GET", "POST"),
        raise_on_status=False,
    )
    adapter = HTTPAdapter(max_retries=retry)
    s.mount("https://", adapter)
    s.mount("http://", adapter)
    return s

def post_with_reconnect(session, payload, attempt_limit=5):
    global url, headers, cookies  # or pass them in
    for attempt in range(attempt_limit):
        try:
            resp = session.post(
                url,
                data=payload,
                headers=headers,
                cookies=cookies,
                timeout=30,
                verify=False,
            )
            resp.raise_for_status()
            return resp
        except (requests.exceptions.ConnectionError,
                requests.exceptions.ChunkedEncodingError) as exc:
            wait = 2 ** attempt
            print(f"Request failed ({exc}); refreshing session, retrying in {wait}s…")
            session.close()
            session = make_session()
            sleep(wait)
    raise RuntimeError("Request still failing after retries")


In [3]:
#------------------------------------------------ Begin_Varible ----------------------------------------

sqldict={'bvdid': [], 'priority': [], 'ListLabel': [], 'Typology': [], 'EntryType': [], 'Name': [], 'InternalID_1': [], 'InternalID_1_type': [], 'InternalID_2': [], 
		  'InternalID_2_type': [], 'InternalID_3': [], 'InternalID_3_type': [], 'CoType': [], 'License_Type': [], 'Address_1': [], 'Address_2': [], 'City': [], 
		  'Zip': [], 'Cntry': [], 'Phone': [], 'Fax': [], 'Website': [], 'Email': [], 'RegulationType': [], 'RegulationTypeCode': [], 'RegulationDate': [], 'CancellationDate': [], 
		  'RegCtry': [], 'RegCode' : [], 'ListCode': [], 'ListLanguage': [], 'ListValidityDate': [], 'ListName': [], 'ListProcessDate': [], 'LEI Code': [], 'BIC SWIFT Code': [], 'Name - Mother Company': [],
		  'Address_1 - Mother company': [], 'Address_2 -  Mother company': [], 'City - Mother company': [], 'Zip - Mother company': [], 'Cntry - Mother company': [], 
		  'Phone - Mother company': [], 'Check': []}

processdate=now.strftime('%Y-%m-%d')


In [4]:
# %%

#------------------------------------------------ Begin_Variable ----------------------------------------

regdict = { 

            'KY CIM 1':'Banking Class A', 

            'KY CIM 2':'Banking Class B', 

            'KY CIM 3':'Money Services', 

            'KY CIM 4':'Trust (Registered PTC)', 

            'KY CIM 5':'Trust (Restricted)',
            'KY CIM 6':'Nominee (Trust)', 

            'KY CIM 7':'Trust (Controlled Subsidiary)', 

            'KY CIM 8':'-', 

            'KY CIM 9':'Company Manager', 

            'KY CIM 10':'Corporate Service Provider', 
            'KY CIM 11':'Full List of all Insurance Entities Registered with the Cayman Islands Monetary Authority', 
            'KY CIM 12':'List of all Mutual Funds registered/licensed with the Cayman Islands Monetary Authority', 

            'KY CIM 13':'List of all Mutual Fund Administrators licensed with the Cayman Islands Monetary Authority', 

            'KY CIM 14':'Securities - Registered Person', 

            'KY CIM 15':'Private Fund', 

            'KY CIM 16':'Virtual Asset Service Provider Registration', 

            'KY CIM 17':'Securities - Full', 

            'KY CIM 18':'Building Society',
            'KY CIM 19':'Credit Union',

            'KY CIM 20':'Development Bank', 


            }


Typology =  { 

            'KY CIM 1':'Bank List - Category A Banks', 

            'KY CIM 2':'Bank List - Category B Banks', 

            'KY CIM 3':'Money Services Providers', 

            'KY CIM 4':'List of (unrestricted) Trust Companies licensed with the Cayman Islands Monetary Authority', 

            'KY CIM 5':'List of Restricted Trust Companies Licensed with the Cayman Islands Monetary Authority', 

            'KY CIM 6':'List of Nominee Companies licensed with the Cayman Islands Monetary Authority', 

            'KY CIM 7':'List of Controlled Subsidiaries registered with the Cayman Islands Monetary Authority', 

            'KY CIM 8':'List of PTCs Registered with the Cayman Islands Monetary Authority', 

            'KY CIM 9':'List of Company Managers Licensed with the Cayman Islands Monetary Authority', 

            'KY CIM 10':'List of Corporate Service Providers Licensed with the Cayman Islands Monetary Authority', 
            'KY CIM 11':'Full List of all Insurance Entities Registered with the Cayman Islands Monetary Authority', 
            'KY CIM 12':'List of all Mutual Funds registered/licensed with the Cayman Islands Monetary Authority', 

            'KY CIM 13':'List of all Mutual Fund Administrators licensed with the Cayman Islands Monetary Authority', 

            'KY CIM 14':'Securities - Registered Person', 

            'KY CIM 15':'Private Fund', 

            'KY CIM 16':'Virtual Asset Service Provider Registration', 

            'KY CIM 17':'Securities - Full', 

            'KY CIM 18':'Building Society',
            'KY CIM 19':'Credit Union',

            'KY CIM 20':'Development Bank', 


            }

In [5]:

url = "https://www.cima.ky/search-entities-cima/get_search_data"

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,image/apng,*/*;q=0.8,application/signed-exchange;v=b3;q=0.7",
    "Accept-Language": "en-US,en;q=0.9",
    "Content-Type": "application/x-www-form-urlencoded",
    "Origin": "https://www.cima.ky",
    "Referer": "https://www.cima.ky/search-entities-cima",
}
cima_cfrf_token_cookie_name = "98a6b29ad78ef9aa443b24bac4d3d35d"
cima_cfrf_token_name = '98a6b29ad78ef9aa443b24bac4d3d35d'
phpsessid = "cae22e2f40343496e42d5688cad00eda"
g_recaptcha_response = "0cAFcWeA7QWtIH0G_MZ9_x0hgBtQbxYiPIduD4km0QyyGLMCR-eYxLend76SCFPM3zQUEbqpTLlHwAo1tgescQOJc-bh7YrCxC37-3VoCYQnvoSJ22R4BsJ8UyjSCU_k6pWd5WegwIGwZ9-BpC_BTTeO-E52Dwy3d_vBXIl_YQVjY-ZJZ-73gaxlYLMytWCe0iddURABAyrqs39iblIyrkh2r-2zwWQUO3G90xMMBxhNiHm7idicK_41U1ZfBsal7dZs1tfuDDrbBNVBxLal-BzfDPRSfl6POKsmGy3m4fUZs60OG8GebFYLRhpkbf02zcY9V8zO6CpCaUyF7YNYgeOqTiWkrNlmjGjkewtyW7WRQscMpNyOGzF58_3kAqRGC3wIOumSoksN5gUSFU1ZowkxxGzfw2___vlrSXvfJOg09J5_J6MB_DaKshgWmXia3NJe5FAodUEu3wSfUYYZP8yfCCKo2hB3Wz5o9zJkivNtjLvmqhkw4RAoPXqLsMI_VW1FwM5SrzTYPdgdM1G-E3fMWG3F4Jn0sc5qHCNgreUVZuMJC1oXTWKMh0V17Nuq4-bCrsoUPr8QBPVkzSfNzXmDZIEdK93NDR-5scHuwzycF29N61bFncmt3_VlDDcEMQ2AE7ybcmv5nIxJ9xEEgCLVJDIYF6bdZu2bBUBCIatw1Cbe3mVn6kQocrmu8z-wp1xXYAKrk0-daGnvH_pvEtSjN-lXGY39Y7yUOuke2kfskZlSy_rhR718RKV3wnCS3uWUm8jU-bFUjZJ0sI1eP_psZgz1ba3B990DMvE8Ok4i4ab8wWRpUsk52qxUIYxcuABPsENq_6D4cHREvwiJ_l0xEYSqtJBB8nSCrtrU0QhBV8rtHCWYDl-CEh9SnKQecRm_yUn9z0Vmzgqo5FW6otMWk0S3TgZ_BXH7gqzBB61Mu3qqC5o-tpfyF0mFqZndsVQFJwHT79BdViV8q0E0ukdGF0zU8qXRnf9rhIVi35uLyC78AGmolKLuBBPMRIXqONPfmza3O-HhZT-9-YNFS5naHld-9aWeeg_sZrq_fh7S6lI47Izz6aPnpmmFJOQu9CSPPWsx0kxdniyutYZ1T_ATKht0h0rmhXWARSaKm_KWboe8WmosThMoyFrZ3mf0AnjwntEJf6-gSdGobp9gCdLpgyzQciyzFGbDqVXmEjGxJquKBtzzixMQB3xSxAxxvLLLcP0IBZyvHEDWwumu2gIT7s789ft4IcTpn70LyqDPVHEPgJtg2wx7zEUbZtCIbN-xY3RQDjSi7OmtHNr2GEkOPoq1-7aGOd9pzMeUBdmzVIdQ8dRC3IJn_UDXglGABxyW0e-fLBv3co3JaSBuOQ8_QpnYAkJjuj-HhMmcQeNdM2rfo0xt8WfP6w3ePT_ZowBEkjxGPfbjmRwAWhUGFplRr0tpiI7dXc6OsSQgXb5k3SDv52r7EQm7UYq7AJ0EcduPfog6rS3vIbkqKLjYNmK_9kAiL0sPXjEMBVBUeRfx_q4wRwOAu6GD6EPj1FLNCEAIs2IPFIxnQfGlq-uoQ7K7mx8ug62E1tliiv4BGNdipbRlIy5pzdpQknIk_u2Dn9ryNlEKSTE3W-WEclcRZWEyctaTyWODsVBwe0QaIqO4pwQJ07PLKSOITwcHOeT4eHRZLejURTYvdf5dEtfNF0hBp-vXvUZdPz9-hS1o193n-zcjwZpytu2-zH3Jqe0IAEmLXxaIOwRk8FL6nQIg9K2YmYJLf-j7PfEkJRZOSEOs8dhKWq-9hVc9mWuZqYtBJf65XzUllez3bGk1Gq-RSQX8eZmPDCNm-DLtDE9j10jQxOLqtSNBoCw_CA1Po8_nPrWNEtuLZZ7LLPa6eFham11yGtHfADSMzssc7qQgTAxmYtG-WedmOtYFglbrz0twes0TWRra2J7SISi1gX1uwlrk15Cj7YQBKDzMq05pTb_iY2Glrv-6dr_1WONfisDV_UULQA2mlLxzrWqJPcSHY3S1hKsL9g3GYm9UQP7tx0D_4616omE-G91Akl8blVCFawwoDpV6sp44OzlAUiQIx9QyxBjnM-NfPqi6876Vq9w17WZZUr5BsxN7uC0NOtNfppg7PjfaCjvNJeONqQHCQepOzLondX1vjP9fmvs9JQy2OyW9w02TgR3938UmU4smQb91bZgFu47X3doqIX-0RMf69v-3LfktpMtRyH-xoNzBzJfcwlw2AFvjZ6ZauAIu6azFaUUvc3M4iAnO-MN7jS12aaYg9Vg4A-bg8D16Gb5jIy9igdMuXm2dFrz11ur0ZaaoWAPOjegIvV"


cookies = {
    "cima_cfrf_token_cookie_name": cima_cfrf_token_cookie_name,
    "PHPSESSID": phpsessid,
    "popup": "Y",
}

session = requests.Session()



In [6]:
# token keep 24h alive, change everytime
base_payload = {
    "cima_cfrf_token_name": cima_cfrf_token_name,
    "Searching": "",
    #"AuthorizationType": "All",
    'AuthorizationType':'Mutual Fund Administrator - Restricted',
    "g-recaptcha-response": g_recaptcha_response,
    "hiddenRecaptcha": "",  # include if the form has it
    "p": "y",
    "ajax": "Y",
}

response = session.post(url, data=base_payload, headers=headers, cookies=cookies, timeout=30,verify=False)


if response.status_code != 200:
    raise RuntimeError(f"Page 1 failed: HTTP {response.status_code}")

# --- First Page ---
soup = BeautifulSoup(response.text, "html.parser")
table = soup.select_one("div.table-responsive")
rows = table.select("tr") if table else []
print(f"Fetching 1 page...")
for row in rows[1:]:
    cells = [td.get_text(strip=True) for td in row.find_all("td")]
    internal_id = cells[0]
    name_ = cells[1]
    type_ = cells[2]
    regulated_date = cells[3]
    sqldict['Name'].append(name_)
    sqldict['Typology'].append(type_)
    sqldict['InternalID_1'].append(internal_id)
    sqldict['InternalID_1_type'].append('Reference Number')
    sqldict['ListProcessDate'].append(processdate)

    sqldict['RegulationDate'].append(regulated_date)
    #sqldict['ListName'].append(Typology[reg])
    sqldict['RegulationType'].append('Regulated')
    # sqldict['RegCtry'].append(reg.split(' ')[0])
    # sqldict['RegCode'].append(reg.split(' ')[1])
    # sqldict['ListCode'].append(reg.split(' ')[-1])
sqldict = bourange_same_length_array(sqldict)

# --- The rest pages ---
while True:
    page_list = soup.find_all("div", class_="pagination-wrap")
    next_li = soup.select_one("li#last a.last")
    if not next_li:
        print("Reached final page.")
        break
    onclick = next_li.get("onclick", "")
    match = re.search(r"PageNumber=(\d+)", onclick)
    if not match:
        print("Next link missing PageNumber; stopping.")
        break
    next_page = int(match.group(1))
    print(f"Fetching page {next_page}...")
    payload = {
            "PageNumber": next_page,
            "p": 'y',
            "ajax": "Y",
            "cima_cfrf_token_name": cima_cfrf_token_name,
            "Searching": "",
            'AuthorizationType':'Mutual Fund Administrator - Restricted',
            }
    sleep(1)
    #resp = session.post(url, data=payload, headers=headers, cookies=cookies, timeout=30, verify=False)
    resp = post_with_reconnect(session, payload)
    if resp.status_code != 200:
        raise RuntimeError(f"Page {next_page} failed: HTTP {resp.status_code}")
    sleep(1)
    soup = BeautifulSoup(resp.text, "html.parser")
    table = soup.select_one("div.table-responsive")
    rows = table.select("tr") if table else []

    for row in rows[1:]:
        cells = [td.get_text(strip=True) for td in row.find_all("td")]
        internal_id = cells[0]
        name_ = cells[1]
        type_ = cells[2]
        regulated_date = cells[3]
        sqldict['Name'].append(name_)
        sqldict['Typology'].append(type_)
        sqldict['InternalID_1'].append(internal_id)
        sqldict['InternalID_1_type'].append('Reference Number')
        sqldict['ListProcessDate'].append(processdate)

        sqldict['RegulationDate'].append(regulated_date)
        #sqldict['ListName'].append(Typology[reg])
        sqldict['RegulationType'].append('Regulated')
        # sqldict['RegCtry'].append(reg.split(' ')[0])
        # sqldict['RegCode'].append(reg.split(' ')[1])
        # sqldict['ListCode'].append(reg.split(' ')[-1])
    sqldict = bourange_same_length_array(sqldict)



Fetching 1 page...
Reached final page.


In [10]:
# %%

#------------------------------------------------ Begin_writer and save df to excel  ----------------------------------------
df=pd.DataFrame(sqldict)


In [12]:
df

,bvdid,priority,ListLabel,Typology,EntryType,Name,InternalID_1,InternalID_1_type,InternalID_2,InternalID_2_type,...,LEI Code,BIC SWIFT Code,Name - Mother Company,Address_1 - Mother company,Address_2 - Mother company,City - Mother company,Zip - Mother company,Cntry - Mother company,Phone - Mother company,Check
0,,,,Mutual Fund Administrator - Restricted,,ALEXANDRIA GLOBAL INVESTMENT MANAGEMENT LTD.,3680,Reference Number,,,...,,,,,,,,,,
1,,,,Mutual Fund Administrator - Restricted,,Five Continents Financial Limited,1172113,Reference Number,,,...,,,,,,,,,,
2,,,,Mutual Fund Administrator - Restricted,,Icatu Finance and Investments Inc.,3737,Reference Number,,,...,,,,,,,,,,
3,,,,Mutual Fund Administrator - Restricted,,Prodigy Asset Management (Cayman) Co.,5966,Reference Number,,,...,,,,,,,,,,
4,,,,Mutual Fund Administrator - Restricted,,"VBT Bank & Trust, Ltd.",3026,Reference Number,,,...,,,,,,,,,,


In [8]:
# Count occurrences per typology (don't call groupby() without 'by' or 'level')
typology_counts = df['Typology'].value_counts(dropna=False)
typology_counts

Typology
Mutual Fund Administrator - Restricted    5
Name: count, dtype: int64

In [9]:
mapping = {
    "Banking Class A": ("1", "Banking Class A"),
    "Banking Class B": ("2", "Banking Class B"),
    "Money Services": ("3", "Money Services"),
    "Trust": ("4", "Trust"),
    "Trust (Restricted)": ("5", "Trust (Restricted)"),
    # also regulated type is regulated?
    "Nominee (Trust)": ("6", "Nominee (Trust)"),
    "Trust (Controlled Subsidiary)": ("7", "Trust (Controlled Subsidiary)"),
    "Trust (Registered PTC)": ("8", "Trust (Registered PTC)"),
    "Company Manager": ("9", "Company Manager"),
    "Corporate Service Provider": ("10", "Corporate Service Provider"),
    
    "Class A Local Insurer":("11","Insurance Entities"),
    "Class A External Insurer":("11","Insurance Entities"),
    "Class B Insurer":("11","Insurance Entities"),
    "Class C Insurer":("11","Insurance Entities"),
    "Class D Insurer":("11","Insurance Entities"),
    "Insurance Agent":("11","Insurance Entities"),
    "Insurance Broker":("11","Insurance Entities"),
    "Insurance Manager":("11","Insurance Entities"),
    "Portfolio Insurance Company":("11","Insurance Entities"),
    
    "Mutual Fund - Registered": ("12", "Mutual Fund - Registered/Licenced"),
    "Mutual Fund - Licenced": ("12", "Mutual Fund - Registered/Licenced"),
    "Mutual Fund - Administered": ("13", "Mutual Fund - Administered"),
    "Securities - Registered Person": ("14", "Securities - Registered Person"),
    "Private Fund": ("15", "Private Fund"),
    "Virtual Asset Service Provider Registration": ("16", "Virtual Asset Service Provider Registration"),
    "Securities - Full": ("17", "Securities - Full"),
    "Building Society": ("18", "Building Society"),
    "Credit Union": ("19", "Credit Union"),
    "Development Bank": ("20", "Development Bank"),
}

mapped = df["Typology"].map(mapping).apply(pd.Series)
mapped.columns = ["ListCode", "ListName"]
df[["ListCode", "ListName"]] = mapped
df['ListProcessDate'] = processdate
df['RegCtry'] = 'KY'
df['RegCode'] = 'CIMA'
df['RegulationType'] = df['RegulationType'].fillna('Regulated')
df = (
    df.assign(ListCode=df['ListCode'].fillna('').astype(str).str.strip())
      .loc[lambda d: d['ListCode'] != '']
)

ValueError: Length mismatch: Expected axis has 1 elements, new values have 2 elements

In [ ]:
df

,bvdid,priority,ListLabel,Typology,EntryType,Name,InternalID_1,InternalID_1_type,InternalID_2,InternalID_2_type,...,LEI Code,BIC SWIFT Code,Name - Mother Company,Address_1 - Mother company,Address_2 - Mother company,City - Mother company,Zip - Mother company,Cntry - Mother company,Phone - Mother company,Check
0,,,,Mutual Fund Administrator - Restricted,,ALEXANDRIA GLOBAL INVESTMENT MANAGEMENT LTD.,3680,Reference Number,,,...,,,,,,,,,,
1,,,,Mutual Fund Administrator - Restricted,,Five Continents Financial Limited,1172113,Reference Number,,,...,,,,,,,,,,
2,,,,Mutual Fund Administrator - Restricted,,Icatu Finance and Investments Inc.,3737,Reference Number,,,...,,,,,,,,,,
3,,,,Mutual Fund Administrator - Restricted,,Prodigy Asset Management (Cayman) Co.,5966,Reference Number,,,...,,,,,,,,,,
4,,,,Mutual Fund Administrator - Restricted,,"VBT Bank & Trust, Ltd.",3026,Reference Number,,,...,,,,,,,,,,


In [ ]:
df.to_excel(filename, index=False)

sleep(3)

driver.quit()

In [ ]:
df = pd.read_excel('KY CIMA SQL Ready 2025-11-14 14.35.26.xlsx')

In [ ]:
df = (
    df.assign(ListCode=df['ListCode'].fillna('').astype(str).str.strip())
      .loc[lambda d: d['ListCode'] != '']
)

In [ ]:
df['ListCode']

0        12.0
1        15.0
2        15.0
3        14.0
4        15.0
         ... 
33862    15.0
33864    12.0
33865    12.0
33867    12.0
33868    12.0
Name: ListCode, Length: 29998, dtype: object